In [2]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Tugas4-2505060041zuyinamirkham") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [3]:
#A.
df = spark.read.csv(
    "hdfs://localhost:9000/user/irkham/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)
print("Tipe objek", type(df))
df.printSchema()

df.show(10)
print("jumlah baris", df.count())

Tipe objek <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)



+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
|ORD-3005|2026-09-09 00:00:00|             Fashion| Purworejo|           5|       20000|

[Stage 3:>                                                          (0 + 1) / 1]

jumlah baris 1000


In [6]:
#B.
from pyspark.sql.functions import col, count, avg

print("jumlah nilai kosong di rating: ", df.filter(col("rating").isNull()).count())

df = df.na.fill({"rating": 0})

print("jumlah nilai kosong setelah diisi: ", df.filter(col("rating").isNull()).count())
print("jumlah baris tetap: ", df.count())

jumlah nilai kosong di rating:  0
jumlah nilai kosong setelah diisi:  0
jumlah baris tetap:  1000


Memakai df.na.fill() karena jika memakai df.na.drop() akan menghapus atau menghilangkan keseluruhan data yang mungkin masih berguna sehingga dengan df.na.fill akan mengisi nilai kosong dengan angka default dan mempertahankan ukuran data set aagar tetap utuh untuk proses alasisis selanjutnya

In [7]:
#C.
from pyspark.sql.functions import when

df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
df = df.withColumn (
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))
df.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [11]:
# D.1.
from pyspark.sql.functions import sum as spark_sum, col

pendapatan_tertinggi = df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc())
pendapatan_tertinggi.show()

[Stage 24:>                                                         (0 + 1) / 1]

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [14]:
#D.2
kota_tier_besar = df.filter(col("tier_transaksi") == "Besar").groupBy("kota").agg(count("order_id").alias("jumlah_transaksi")) \
    .orderBy(col("jumlah_transaksi").desc())
kota_tier_besar.show()

[Stage 28:==========================================================(1 + 0) / 1]

+----------+----------------+
|      kota|jumlah_transaksi|
+----------+----------------+
|      Solo|              92|
|  Magelang|              78|
|   Kebumen|              78|
|Yogyakarta|              75|
| Purworejo|              66|
|  Semarang|              65|
+----------+----------------+



In [15]:
#D.3
rating_pembayaran = df.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc())
rating_pembayaran.show()

[Stage 31:>                                                         (0 + 1) / 1]

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [16]:
#E
df.write.mode("overwrite").option("header", "true").csv(
    "hdfs://localhost:9000/user/irkham/tugas4/pengolahan_data_transaksi"
)

print("berhasil disimpan di HDFS")
!hdfs dfs -ls /user/irkham/tugas4/pengolahan_data_transaksi

berhasil disimpan di HDFS
Found 2 items
-rw-r--r--   3 irkham supergroup          0 2026-09-12 18:50 /user/irkham/tugas4/pengolahan_data_transaksi/_SUCCESS
-rw-r--r--   3 irkham supergroup      97296 2026-09-12 18:50 /user/irkham/tugas4/pengolahan_data_transaksi/part-00000-1133b0f7-765c-496d-aed8-5ceefc577801-c000.csv


spark menyimpan hasil dalam beberapa file partisipasi seprti part-00000,bukan satu file tunggal.Karena spark memproses data secara terdistribusi berdasarkan partisipasi.Setiap partisipasi ditulis oleh executor yang berbeda sehingga proses penyimpanan dapat dilakukan secara pararel tanpa menggangu partisipasi lain.